# FC코드 → 메뉴명 매핑 추출

Drive `finetune_dataset/` 이미지 파일명에서 메뉴명을 추출합니다.

**파일명 형식:** `A_13_A13001_가자미구이_17_01.jpg`
- 3번째 필드: Keycode (A13001)
- 4번째 필드: 메뉴명 (가자미구이)

**출력:** `fc_to_menu_map.yaml` — FC코드 → 메뉴명 리스트

In [ ]:
# Step 1: Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Step 2: 이미지 파일명에서 메뉴명 추출
import os
from pathlib import Path
from collections import defaultdict

DATASET_DIR = Path('/content/drive/MyDrive/finetune_dataset')

# train + val 모두 스캔
img_dirs = [
    DATASET_DIR / 'images' / 'train',
    DATASET_DIR / 'images' / 'val',
]

# 실제 경로 확인
for d in img_dirs:
    exists = d.exists()
    count = len(list(d.iterdir())) if exists else 0
    print(f"{d}: {'OK' if exists else 'NOT FOUND'} ({count}건)")

In [ ]:
# Step 3: 파일명 파싱 → keycode → 메뉴명 매핑
#
# 파일명 형식: A_13_A13001_가자미구이_17_01.jpg
#              대분류_중분류_Keycode_음식명_표본번호_촬영각도.확장자

keycode_to_menu = {}      # A13001 → 가자미구이
fc_to_menus = defaultdict(set)  # 나중에 FC코드 매핑용
menu_count = defaultdict(int)   # 메뉴별 이미지 수
parse_errors = []

for img_dir in img_dirs:
    if not img_dir.exists():
        continue
    for fname in os.listdir(str(img_dir)):
        if not fname.lower().endswith(('.jpg', '.jpeg', '.png')):
            continue
        parts = fname.rsplit('.', 1)[0].split('_')
        # 최소 4개 필드: 대분류_중분류_Keycode_음식명
        if len(parts) < 4:
            parse_errors.append(fname)
            continue
        
        keycode = parts[2]    # A13001
        menu_name = parts[3]  # 가자미구이
        
        keycode_to_menu[keycode] = menu_name
        menu_count[menu_name] += 1

print(f"고유 Keycode 수: {len(keycode_to_menu)}")
print(f"고유 메뉴명 수: {len(set(keycode_to_menu.values()))}")
print(f"총 이미지 수: {sum(menu_count.values())}")
print(f"파싱 실패: {len(parse_errors)}건")
if parse_errors[:5]:
    print(f"  예시: {parse_errors[:5]}")

In [ ]:
# Step 4: YOLO class_id ↔ FC코드 매핑 복원
#
# prepare_local_dataset.py의 collect_classes()가 JSON의 food_type.fc를 사용했음
# JSON이 없으면 labels/*.txt + 이미지 파일명으로 역매핑
#
# labels txt 형식: class_id cx cy w h
# 같은 파일명의 이미지 → 메뉴명, 라벨 → class_id

label_dirs = [
    DATASET_DIR / 'labels' / 'train',
    DATASET_DIR / 'labels' / 'val',
]

class_id_to_menus = defaultdict(set)  # class_id → {메뉴명1, 메뉴명2, ...}
class_id_to_count = defaultdict(lambda: defaultdict(int))  # class_id → {메뉴명: 이미지수}

for img_dir, label_dir in zip(img_dirs, label_dirs):
    if not label_dir.exists():
        continue
    for label_file in os.listdir(str(label_dir)):
        if not label_file.endswith('.txt'):
            continue
        
        stem = label_file.rsplit('.', 1)[0]
        # 같은 이름의 이미지 찾기
        parts = stem.split('_')
        if len(parts) < 4:
            continue
        menu_name = parts[3]
        
        # 라벨 파일에서 class_id 읽기
        label_path = label_dir / label_file
        with open(label_path) as f:
            for line in f:
                tokens = line.strip().split()
                if tokens:
                    cid = int(tokens[0])
                    class_id_to_menus[cid].add(menu_name)
                    class_id_to_count[cid][menu_name] += 1
                    break  # 한 이미지에 보통 1개 객체

print(f"class_id 수: {len(class_id_to_menus)}")
print(f"\n클래스별 메뉴 수:")
for cid in sorted(class_id_to_menus.keys()):
    menus = class_id_to_menus[cid]
    print(f"  class_{cid}: {len(menus)}종 — {', '.join(sorted(menus)[:5])}{'...' if len(menus) > 5 else ''}")

In [ ]:
# Step 5: YAML 생성 — class_id → {category, menus}
import yaml

# food_class_names.yaml의 카테고리명 (로컬에서 복사)
# 없으면 빈 dict로 — class_id만으로도 매핑 생성 가능
CATEGORY_NAMES = {
    0: '빵류', 1: '샌드위치/토스트', 2: '면류(쌀국수/우동)',
    3: '치킨', 4: '피자', 5: '덮밥/포케', 6: '케이크/머핀',
    7: '파스타', 8: '김밥/초밥', 9: '꼬치/스테이크',
    10: '볶음류', 11: '떡', 12: '매운탕/탕류',
    13: '크로와상/타르트', 14: '돈가스/튀김', 15: '식빵/와플',
    16: '샐러드', 17: '만두', 18: '전골/찌개',
    19: '국수/칼국수', 20: '새우튀김/어묵', 21: '전통빵/호빵',
    22: '라멘/라면', 23: '떡볶이', 24: '크림빵/롤',
    25: '짬뽕/짜장', 26: '시리얼/파이', 27: '회/생선회',
    28: '찌개/순두부', 29: '족발/순대', 30: '커피',
    31: '생선구이', 32: '커리/수프', 33: '그라탕/리조또',
    34: '해물볶음', 35: '죽/수프', 36: '곰탕/해장국',
    37: '찜류', 38: '밀크티/라떼', 39: '튀김간식',
    40: '닭발/동파육', 41: '죽/디저트', 42: '도넛/꽈배기',
    43: '비빔밥', 44: '버거', 45: '국밥/떡국',
    46: '두부요리', 47: '나베/전골', 48: '과일주스',
    49: '전/부침개', 50: '에이드', 51: '잎차류',
    52: '중식볶음', 53: '볶음밥', 54: '미역국/기타탕',
    55: '스무디', 56: '옥수수/약밥', 57: '소떡소떡',
    58: '추어전', 59: '어니언링/튀김', 60: '나쵸',
    61: '전통음료', 62: '묵밥', 63: '오믈렛',
    64: '김치찜', 65: '빙수', 66: '은행구이',
    67: '생선조림', 68: '메밀전', 69: '꼬막무침', 70: '기타',
}

mapping = {}
for cid in sorted(class_id_to_menus.keys()):
    cat = CATEGORY_NAMES.get(cid, f'class_{cid}')
    menus_with_count = class_id_to_count[cid]
    # 이미지 수 내림차순 정렬
    sorted_menus = sorted(menus_with_count.items(), key=lambda x: -x[1])
    mapping[cid] = {
        'category': cat,
        'menus': [{'name': m, 'count': c} for m, c in sorted_menus],
    }

# YAML 저장
out_path = '/content/drive/MyDrive/finetune_dataset/fc_to_menu_map.yaml'
with open(out_path, 'w', encoding='utf-8') as f:
    yaml.dump(mapping, f, allow_unicode=True, default_flow_style=False, sort_keys=True)

print(f"저장 완료: {out_path}")
print(f"클래스 수: {len(mapping)}")
total_menus = sum(len(v['menus']) for v in mapping.values())
print(f"총 메뉴 수: {total_menus}")

In [ ]:
# Step 6: 결과 미리보기
print("=" * 60)
print("class_id → 카테고리 → 메뉴명 매핑")
print("=" * 60)
for cid in sorted(mapping.keys()):
    info = mapping[cid]
    cat = info['category']
    menus = info['menus']
    menu_names = [m['name'] for m in menus[:5]]
    suffix = f' ... +{len(menus)-5}종' if len(menus) > 5 else ''
    print(f"  [{cid:>2}] {cat:<18} → {', '.join(menu_names)}{suffix}")

print(f"\n이 파일을 로컬 Object_Detection/config/fc_to_menu_map.yaml 에 복사하세요.")